<a href="https://colab.research.google.com/github/Azira18/audio-restoration-musicgen/blob/main/01_Generate_Audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Generation of audio with MusicGen**
In this notebook we will generate audio with Musicgen. As reported on the [official hugging face website](https://huggingface.co/facebook/musicgen-small), MusicGen, developed by Meta AI, is a model based on the Transformer architecture, the same as models like ChatGPT. It works in an 'autoregressive' way, meaning it builds the music one fragment at a time, relying on what it has already generated. Its main innovation is efficiency: it uses an audio encoder called EnCodec to translate music into digital 'tokens'. Unlike other models, MusicGen generates these tokens in a single step and, thanks to a parallelization technique, it can create one second of audio in just 50 computation steps, making it relatively fast. The system was trained to work with 32 kHz audio.

## Import and Utilities

In [ ]:
# =================================================================
#                 GOOGLE COLAB ENVIRONMENT SETUP
# =================================================================
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/audio-restoration-musicgen'
os.makedirs(PROJECT_PATH, exist_ok = True)
INPUT_FOLDER_ABSOLUTE = os.path.join(PROJECT_PATH, 'data/generated/')
os.makedirs(INPUT_FOLDER_ABSOLUTE, exist_ok = True)


# ==============================
#             IMPORT
# ==============================
!pip install -q transformers accelerate

import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import os
import warnings
from IPython.display import Markdown, display, Audio
import soundfile as sf
import numpy as np
import random
warnings.filterwarnings("ignore")

# ==========================================================
#              VERIFYNG IF GPU IS AVAILABLE
# ==========================================================
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'



Mounted at /content/drive


## Setup for generation

In [ ]:
###### LOADING PROCESSOR AND MODEL FROM HUGGING FACE🤗 ######

processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")
model.to(device)

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

MusicgenForConditionalGeneration(
  (text_encoder): T5EncoderModel(
    (shared): Embedding(32128, 768)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 768)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=768, out_features=768, bias=False)
                (k): Linear(in_features=768, out_features=768, bias=False)
                (v): Linear(in_features=768, out_features=768, bias=False)
                (o): Linear(in_features=768, out_features=768, bias=False)
                (relative_attention_bias): Embedding(32, 12)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=768, out_features=3072, bias=False)
                (wo): L

In [ ]:
###### DEFINING PROMPTS ######

prompt_categories  = {
    "rock": [
        "Energetic rock track with powerful drums, punchy electric bass, distorted electric guitar riffs, and subtle piano chords in the background.",
        "Classic rock ballad with steady drum groove, melodic bass line, clean guitar arpeggios, and expressive piano.",
        "Fast-paced punk rock with aggressive drums, heavy bass, crunchy rhythm guitar, and minimal piano accent.",
        "Progressive rock jam with complex drum fills, groovy bass, psychedelic guitar solos, and bright piano chords."
    ],
    "lofi": [
        "Lo-fi chill track with soft dusty drums, warm bass, jazzy electric guitar, and dreamy piano chords.",
        "Relaxed hip-hop lo-fi beat with laid-back drums, deep bass, muted guitar plucks, and lo-fi piano samples.",
        "Ambient lo-fi groove with minimal drums, smooth bass, reverb guitar textures, and mellow piano melodies."
    ],
    "jazz_blues": [
        "Jazz quartet with acoustic drums, walking double bass, bluesy electric guitar solos, and improvising piano.",
        "Smooth jazz groove with brushes on drums, upright bass, clean jazz guitar chords, and melodic piano lines.",
        "Blues shuffle with swing drums, strong bass, overdriven guitar licks, and boogie-woogie piano."
    ],
    "funk_soul_disco": [
        "70s funk groove with tight drums, slap bass, wah-wah rhythm guitar, and bright electric piano.",
        "Soulful track with groovy drums, warm bass, clean guitar strumming, and Rhodes piano chords.",
        "Disco beat with steady four-on-the-floor drums, funky bass, rhythmic guitar, and glittering piano."
    ],
    "edm_electro": [
        "Electronic house track with punchy kick drums, deep sub bass, electric guitar stabs, and percussive piano riff.",
        "Synthwave groove with retro drums, dark driving bass, clean guitar melodies, and bright synth-piano chords.",
        "Lo-fi house beat with crunchy drums, minimal bass, guitar samples, and repetitive piano loop."
    ],
    "cinematic_experimental": [
        "Cinematic score with dramatic drums, deep bass, atmospheric guitar textures, and emotional piano.",
        "Minimalist experimental piece with sparse drums, sub bass, processed guitar, and prepared piano.",
        "Epic orchestral rock fusion with pounding drums, distorted bass, electric guitar power chords, and grand piano."
    ]
}

def pick_random_prompts(n=4):
    '''
    Args:
        number (int) of audio we want to generate

    Returns:
        n prompts of different categories
    '''
    categories = random.sample(list(prompt_categories.keys()), n)  # n different categories
    selected_prompts = [random.choice(prompt_categories[cat]) for cat in categories]
    return selected_prompts

prompts = pick_random_prompts()
print(prompts)


['Cinematic score with dramatic drums, deep bass, atmospheric guitar textures, and emotional piano.', 'Blues shuffle with swing drums, strong bass, overdriven guitar licks, and boogie-woogie piano.', 'Ambient lo-fi groove with minimal drums, smooth bass, reverb guitar textures, and mellow piano melodies.', '70s funk groove with tight drums, slap bass, wah-wah rhythm guitar, and bright electric piano.']


In [ ]:
###### SETTING AUDIO DURATION ######

duration_in_seconds = 10
duration_in_tokens = int(duration_in_seconds * 50)

# MusicGen uses an encoding-decoding model called EnCodec, which compresses audio into a
# discrete sequence of tokens and vice versa. According to the official MusicGen
# documentation, EnCodec operates at a frequency of 50 Hz. That means the model generates
# 50 tokens for every second of audio.

In [ ]:
###### DEFINING THE DIRECTORY WHERE WE'RE GOING TO SAVE GENERATED AUDIOS ######

os.makedirs(INPUT_FOLDER_ABSOLUTE, exist_ok=True) # creates the directory if it doesn't exist.

## Generation

In [ ]:
###### CREATING AND SAVING AUDIOS ######

if len(os.listdir(INPUT_FOLDER_ABSOLUTE)):
    for file in os.listdir(INPUT_FOLDER_ABSOLUTE):
        file_path = os.path.join(INPUT_FOLDER_ABSOLUTE, file)
        if os.path.isfile(file_path):
            os.remove(file_path)

for i, prompt in enumerate(prompts):

    print(f'\ngenerating sample {i+1}/{len(prompts)}...')

    # processing prompts
    inputs = processor(
        text = [prompt],       # the text it has to translate
        padding = True,        # not usefull in this specific case, it's only for more robust code. It's used to make all prompts the same
                               # length if we pass more than one at a time, so that it can return a rectangular tensor.
        return_tensors = 'pt'  # we're saying "return a Pytorch Tensor please!"
    ).to(device)

    # generating audios
    audio_values = model.generate(**inputs, max_new_tokens=duration_in_tokens)

    # preparing audios for saving
    sampling_rate = model.config.audio_encoder.sampling_rate # It accesses model's technical data sheet (config),
                                                             # navigates to the audio_encoder section and reads the
                                                             # sampling_rate at which the model's audio should be interpreted.
                                                             # For Musicgen-small this value is 32 kHz.

    audio_array = audio_values.cpu().numpy().squeeze() # Pytorch tensor is converted to a NumPy array as that's the format
                                                       # understood by SciPy. Each number of this array represents the amplitude
                                                       # of the sound wave at a specific moment in time.

    # saving audios
    peak = np.max(np.abs(audio_array))
    if peak > 0:
        audio_array_normalized = audio_array / peak * 0.98  # Normalize and leave some headroom
    else:
        audio_array_normalized = audio_array # Avoid division by zero if the audio is silent


    prompt_name_as_file = prompt.lower().replace(" ", "_").split('_with')[0]
    output_path = os.path.join(INPUT_FOLDER_ABSOLUTE, f"sample_{i+1}_{prompt_name_as_file}.wav")

    sf.write(output_path, audio_array_normalized, sampling_rate)
        # That literally means "take this data (audio_array_normalized) and save the file in this directory (output_path) by using this audio quality
        # (sampling_rate)"

        # A .wav file is like a container that holds not only the list of numbers (audio_array) but also essential information
        # needed to interpret it, including the sampling rate: when the audio player reads this value it understands that to reproduce the
        # audio, it has to read (for example) 32.000 numbers from the array each second and send them to the speaker.


    print(f'sample {i+1} generated and saved successfully in data/generated/{prompt_name_as_file}.wav')
    display(Audio(data=audio_array_normalized, rate=sampling_rate))

print(f'\nsampling rate: {sampling_rate}Hz')


generating sample 1/4...
sample 1 generated and saved successfully in data/generated/cinematic_score.wav



generating sample 2/4...
sample 2 generated and saved successfully in data/generated/blues_shuffle.wav



generating sample 3/4...
sample 3 generated and saved successfully in data/generated/ambient_lo-fi_groove.wav



generating sample 4/4...
sample 4 generated and saved successfully in data/generated/70s_funk_groove.wav



sampling rate: 32000Hz
